In [ ]:
# Importing all the tools we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Makes charts look clean and professional
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

print("Libraries loaded successfully!")

In [ ]:
# Load the CSV file into a DataFrame (like an Excel table in Python)
df = pd.read_csv('data/london_merged.csv')

# See the first 5 rows
df.head()

In [ ]:
# How many rows and columns?
print("Shape:", df.shape)

# What are the column names?
print("\nColumns:", df.columns.tolist())

# What data types are in each column?
print("\nData types:")
print(df.dtypes)

# Any missing values?
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Rename columns to be more readable
df.rename(columns={
    'timestamp': 'date',
    'cnt': 'bike_rides',
    't1': 'temp_real_c',
    't2': 'temp_feels_like_c',
    'hum': 'humidity_percent',
    'wind_speed': 'wind_speed_kph',
    'weather_code': 'weather',
    'is_holiday': 'is_holiday',
    'is_weekend': 'is_weekend',
    'season': 'season'
}, inplace=True)

# Convert date column to proper datetime format
df['date'] = pd.to_datetime(df['date'])

# Extract useful time parts from the date
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# Map season numbers to readable names
df['season'] = df['season'].map({
    0: 'Spring', 1: 'Summer', 2: 'Autumn', 3: 'Winter'
})

# Map weather codes to readable labels
df['weather'] = df['weather'].map({
    1: 'Clear', 2: 'Scattered Clouds', 3: 'Broken Clouds',
    4: 'Cloudy', 7: 'Rain', 10: 'Rain with Thunderstorm', 26: 'Snowfall'
})

print("Data cleaned successfully!")
print(df.head())

In [ ]:
# Group data by season and calculate average rides
season_avg = df.groupby('season')['bike_rides'].mean().sort_values(ascending=False).reset_index()

# Plot it
plt.figure(figsize=(10, 5))
sns.barplot(data=season_avg, x='season', y='bike_rides', palette='Blues_d')
plt.title('Average Bike Rides by Season', fontsize=16, fontweight='bold')
plt.xlabel('Season')
plt.ylabel('Average Number of Rides')
plt.tight_layout()
plt.savefig('season_analysis.png', dpi=150)
plt.show()

print(season_avg)

In [ ]:
# Average rides per hour
hourly_avg = df.groupby('hour')['bike_rides'].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=hourly_avg, x='hour', y='bike_rides', marker='o', color='steelblue', linewidth=2.5)
plt.title('Average Bike Rides by Hour of Day', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day (0 = midnight, 8 = 8am)')
plt.ylabel('Average Number of Rides')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig('hourly_analysis.png', dpi=150)
plt.show()

In [ ]:
# Order days correctly
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

day_avg = df.groupby('day_of_week')['bike_rides'].mean().reindex(day_order).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=day_avg, x='day_of_week', y='bike_rides', palette='muted')
plt.title('Average Bike Rides by Day of Week', fontsize=16, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Average Number of Rides')
plt.tight_layout()
plt.savefig('day_analysis.png', dpi=150)
plt.show()

In [ ]:
# Average rides by weather type
weather_avg = df.groupby('weather')['bike_rides'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=weather_avg, x='weather', y='bike_rides', palette='coolwarm')
plt.title('Average Bike Rides by Weather Condition', fontsize=16, fontweight='bold')
plt.xlabel('Weather')
plt.ylabel('Average Number of Rides')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('weather_analysis.png', dpi=150)
plt.show()

In [ ]:
print("=== KEY FINDINGS ===\n")

summer = df[df['season']=='Summer']['bike_rides'].mean()
winter = df[df['season']=='Winter']['bike_rides'].mean()
pct_diff = ((summer - winter) / winter) * 100
print(f"Summer avg rides: {summer:.0f}")
print(f"Winter avg rides: {winter:.0f}")
print(f"Summer vs Winter difference: +{pct_diff:.0f}%")

peak_hour = hourly_avg.loc[hourly_avg['bike_rides'].idxmax(), 'hour']
print(f"\nPeak hour of day: {peak_hour}:00")

busiest_day = day_avg.loc[day_avg['bike_rides'].idxmax(), 'day_of_week']
print(f"Busiest day: {busiest_day}")

best_weather = weather_avg.iloc[0]['weather']
print(f"Best weather for cycling: {best_weather}")

In [ ]:
## Finding 1: Seasonal Demand
Summer months see significantly higher ridership compared to winter.
This suggests TfL should increase bike availability and maintenance
cycles ahead of spring/summer to meet demand.

## Finding 2: Peak Hours
Demand peaks at 8am and 5-6pm, clearly driven by commuter patterns.
Weekend peaks shift to midday, suggesting leisure usage.